# FGSM Attack Implementation
First, let's install the required packages.

In [1]:
%pip install transformers torch datasets


[notice] A new release of pip available: 22.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import Dependencies

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/lhinds/repos/experimental-projects/FGSM_LLM_Attack/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Loading Function

In [3]:
def load_model(model_name="gpt2"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()  # Set to evaluation mode
    return model, tokenizer

## Loss Generation Function

In [4]:
def generate_loss(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]  # Ensure padding is handled
    embeddings = model.get_input_embeddings()(input_ids).detach()
    embeddings.requires_grad_()

    labels = input_ids.clone()
    outputs = model(inputs_embeds=embeddings, labels=labels, attention_mask=attention_mask)
    loss = outputs.loss

    return loss, embeddings, attention_mask  # Return attention_mask as well


## FGSM Attack Implementation

In [5]:
def apply_fgsm_attack(embeddings, loss, epsilon):
    loss.backward()  # Compute gradients
    perturbation = epsilon * embeddings.grad.sign()  # FGSM perturbation
    perturbed_embeddings = embeddings + perturbation  # Add perturbation
    return perturbed_embeddings


## Output Comparison Function

In [6]:
def compare_outputs(model, tokenizer, original_text, perturbed_embeddings, attention_mask):
    # Generate original output
    original_ids = tokenizer(original_text, return_tensors="pt", padding=True, truncation=True)
    original_output = tokenizer.decode(
        model.generate(**original_ids, pad_token_id=tokenizer.eos_token_id)[0]
    )

    # Generate adversarial output using perturbed embeddings
    perturbed_output = tokenizer.decode(
        model.generate(
            inputs_embeds=perturbed_embeddings,
            attention_mask=attention_mask,  # Ensure correct token handling
            pad_token_id=tokenizer.eos_token_id,  # Set PAD token explicitly
            max_new_tokens=50  # Control output length properly
        )[0]
    )

    print("Original Output:", original_output)
    print("Adversarial Output:", perturbed_output)


## Run the Attack
Now let's execute the attack with a sample text.

In [7]:
# Load the model and tokenizer
model, tokenizer = load_model()

# Fix: Define padding token explicitly
tokenizer.pad_token = tokenizer.eos_token  

# Set up the attack parameters
original_text = "The weather today is great."
epsilon = 0.5 # You can modify this value

# Generate the adversarial example
loss, inputs, attention_mask = generate_loss(model, tokenizer, original_text)
perturbed_input = apply_fgsm_attack(inputs, loss, epsilon)

# Compare the results
compare_outputs(model, tokenizer, original_text, perturbed_input, attention_mask)



Original Output: The weather today is great. I'm going to be back in the office for a few days. I'm going to be back
Adversarial Output:  the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
